<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one piece of content, for one client, on one day, over 2026-03-01 to 2026-03-31.

In [1]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

In [2]:
import os
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [12]:
import pandas as pd

HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": HF_TOKEN}
)

dim_content = pd.read_parquet(
    f"{HF_BASE}/dim_content.parquet",
    storage_options={"token": HF_TOKEN}
)

panel_daily = panel_daily.merge(
    dim_content,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    suffixes=("", "_dim")
)
print(panel_daily.shape)

(9841378, 54)


In [5]:
dupe_check = dim_content.duplicated(subset=["client_hash_id","content_hash_id"]).sum()
print("Duplicate keys in dim_content:", dupe_check)

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")
print("dim_content_clean rows:", len(dim_content_clean))


Duplicate keys in dim_content: 0
dim_content_clean rows: 519606


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

panel_daily = panel_daily.merge(
    dim_content_clean,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    suffixes=("", "_dim")
)
print("Post-merge shape:", panel_daily.shape)  # should stay ~9,841,378 rows

print("Row count:", len(panel_daily))



Post-merge shape: (9841378, 78)
Row count: 9841378


In [7]:
n_unique = panel_daily.groupby(["client_hash_id","content_hash_id","report_date"]).ngroups
print("Unique (client, content, date) combos:", n_unique)
print("Any duplicates on that key?", n_unique != len(panel_daily))

print("Date span:", panel_daily["report_date"].min(), "-", panel_daily["report_date"].max())

Unique (client, content, date) combos: 9841378
Any duplicates on that key? False
Date span: 2026-03-01 - 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:**

gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec, content_type, are observed page-day signals for clustering.

**Label:**

none. Clustering is unsupervised.

**Context:**

client_hash_id, content_hash_id, report_date, are identifiers used to trace results back to real content, not used as clustering inputs.

**Excluded:**

AI-referral columns — under 0.1% row coverage this month (verified in Section 3), not usable yet.
fact_content_daily_performance_sample.parquet — sealed final-month test data, never used for logic.
dim_content date fields (last_optimized_date, content_updated_date, optimization_eligible_date) — could leak information about actions taken after the report window.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Duplicate check
print("duplicates on key:", panel_daily.duplicated(subset=["client_hash_id","content_hash_id","report_date"]).any())

# Counts and missingness
feature_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position",
                 "ga4_pageviews","ga4_sessions","ga4_engaged_sessions",
                 "ga4_total_engagement_sec","content_type"]
print("Total rows:", len(panel_daily))
print(panel_daily[feature_cols].isna().sum())

# Availability
available = panel_daily.query("gsc_data_available == True and ga4_data_available == True")
print("Rows before filter:", len(panel_daily))
print("Rows after IS TRUE filter (both GSC and GA4 available):", len(available))

# Window check
print("Max report_date:", panel_daily["report_date"].max(), "— confirm within March 2026, no bleed into sealed month")

duplicates on key: False
Total rows: 9841378
gsc_impressions                   0
gsc_clicks                        0
gsc_avg_position            6230317
ga4_pageviews               3018741
ga4_sessions                3018741
ga4_engaged_sessions        3018741
ga4_total_engagement_sec    3018741
content_type                      0
dtype: int64
Rows before filter: 9841378
Rows after IS TRUE filter (both GSC and GA4 available): 364347
Max report_date: 2026-03-31 — confirm within March 2026, no bleed into sealed month


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Filtering to rows where both GSC and GA4 data are available (IS TRUE) keeps only 364,347 of 9,841,378 rows (3.7%). The rest split as: GSC-only 17.5%, GA4-only 0.5%, and neither source available for 47.7% of rows — meaning roughly half of all page-days in this warehouse carry no performance signal at all for March. Clustering on the fully-available slice would represent a small, non-random subset of content.

gsc_avg_position is null for 6,230,317 rows (63%), and this is fully structural: 100% of those null rows have zero impressions, confirming there's no position to average when a page had no search visibility that day — not a data quality gap.

AI-referral columns are populated in fewer than 0.1% of rows this month, confirming they're too sparse to use as features yet.

This slice covers one month (March 2026) across 55 unique clients — seasonal effects or this client mix won't necessarily generalize to other months or a larger client base.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Availability breakdown
both = panel_daily.query("gsc_data_available == True and ga4_data_available == True")
gsc_only = panel_daily.query("gsc_data_available == True and ga4_data_available == False")
ga4_only = panel_daily.query("gsc_data_available == False and ga4_data_available == True")
neither = panel_daily.query("gsc_data_available == False and ga4_data_available == False")

print("Both available:", len(both), f"({len(both)/len(panel_daily):.1%})")
print("GSC only:", len(gsc_only), f"({len(gsc_only)/len(panel_daily):.1%})")
print("GA4 only:", len(ga4_only), f"({len(ga4_only)/len(panel_daily):.1%})")
print("Neither:", len(neither), f"({len(neither)/len(panel_daily):.1%})")

# gsc_avg_position missingness tied to zero impressions
zero_impressions_null_position = panel_daily.query("gsc_impressions == 0")["gsc_avg_position"].isna().sum()
total_null_position = panel_daily["gsc_avg_position"].isna().sum()
print(f"\nNull gsc_avg_position rows: {total_null_position}")
print(f"Of those, rows with 0 impressions: {zero_impressions_null_position} ({zero_impressions_null_position/total_null_position:.1%})")

# client count sanity check
print("\nTotal unique clients:", panel_daily["client_hash_id"].nunique())

Both available: 364347 (3.7%)
GSC only: 1718348 (17.5%)
GA4 only: 49619 (0.5%)
Neither: 4690323 (47.7%)

Null gsc_avg_position rows: 6230317
Of those, rows with 0 impressions: 6230317 (100.0%)

Total unique clients: 55


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.